# County Rainfall and Planting Advisory

**Analyst:** Marion Oduor · PLP Data Analytics Cohort YOUR COHORT

A farmer-support NGO sends SMS planting advice to five counties. This notebook cleans one hand-typed station record, analyses monthly rainfall with NumPy and produces a planting advisory.

In [1]:
counties = ["Nairobi", "Nakuru", "Kisumu", "Mombasa", "Machakos"]
clean_rows = [
    [64, 56, 92, 219, 176, 35, 18, 24, 31, 61, 165, 104],   # Nairobi
    [35, 42, 70, 145, 120, 78, 96, 105, 72, 55, 60, 38],    # Nakuru
    [58, 90, 150, 240, 180, 95, 70, 85, 90, 88, 130, 95],   # Kisumu
    [30, 15, 55, 150, 240, 110, 90, 65, 60, 90, 100, 70],   # Mombasa
]
machakos_raw = ["48", "35", "88", "N/A", "140", " 20 ",
                "9", "12", "", "45", "182", "96"]

## Task 1: Clean the Machakos Data

**Approach:** `clean_readings()` tries to convert each value to an integer and catches only `ValueError`, recording `None` for anything invalid. Invalid months are then filled with the rounded average of the valid readings, and the imputed months are printed so the change is documented.

In [2]:
months = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
          "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]


def clean_readings(raw_values):
    """Convert text readings to integers; return None in place of any invalid value."""
    cleaned = []
    for value in raw_values:
        try:
            cleaned.append(int(str(value).strip()))
        except ValueError:
            cleaned.append(None)
    return cleaned


parsed = clean_readings(machakos_raw)
valid_values = [v for v in parsed if v is not None]
fill_value = round(sum(valid_values) / len(valid_values))

imputed_months = [months[i] for i, v in enumerate(parsed) if v is None]
machakos_clean = [fill_value if v is None else v for v in parsed]

print(f"Valid readings: {len(valid_values)} of {len(machakos_raw)}")
print(f"Average of valid readings (rounded): {fill_value} mm")
print(f"Imputed months: {', '.join(imputed_months)}")
print("Cleaned Machakos row:", machakos_clean)

Valid readings: 10 of 12
Average of valid readings (rounded): 68 mm
Imputed months: Apr, Sep
Cleaned Machakos row: [48, 35, 88, 68, 140, 20, 9, 12, 68, 45, 182, 96]


## Task 2: Build the 5 × 12 NumPy Array

**Approach:** Add the cleaned Machakos row to the four clean rows and convert the result to a NumPy array. Then check `.shape` and `.dtype`.

In [3]:
import numpy as np

rain = np.array(clean_rows + [machakos_clean])

print("Shape:", rain.shape)
print("Dtype:", rain.dtype)

Shape: (5, 12)
Dtype: int64


## Task 3: Annual Totals and Monthly Averages

**Approach:** `sum(axis=1)` collapses the months to give one total per county. `mean(axis=0)` collapses the counties to give one average per month.

In [4]:
annual_totals = rain.sum(axis=1)
monthly_avg = rain.mean(axis=0).round(1)

print("Annual total rainfall per county:")
for county, total in zip(counties, annual_totals):
    print(f"  {county:<9} {total:>5,} mm")

print("\nAverage rainfall per month across all counties:")
for month, avg in zip(months, monthly_avg):
    print(f"  {month}: {avg:>6.1f} mm")

Annual total rainfall per county:
  Nairobi   1,045 mm
  Nakuru      916 mm
  Kisumu    1,371 mm
  Mombasa   1,075 mm
  Machakos    811 mm

Average rainfall per month across all counties:
  Jan:   47.0 mm
  Feb:   47.6 mm
  Mar:   91.0 mm
  Apr:  164.4 mm
  May:  171.2 mm
  Jun:   67.6 mm
  Jul:   56.6 mm
  Aug:   58.2 mm
  Sep:   64.2 mm
  Oct:   67.8 mm
  Nov:  127.4 mm
  Dec:   80.6 mm


## Task 4: Wettest Month per County

**Approach:** `argmax(axis=1)` returns the position of the largest value in each row. Each position is converted back to a month name, and `zip()` pairs it with its county.

In [5]:
wettest_pos = rain.argmax(axis=1)
wettest_mm = rain.max(axis=1)
wettest_months = [months[p] for p in wettest_pos]

for county, month, mm in zip(counties, wettest_months, wettest_mm):
    print(f"{county:<9} wettest month: {month} ({mm} mm)")

Nairobi   wettest month: Apr (219 mm)
Nakuru    wettest month: Apr (145 mm)
Kisumu    wettest month: Apr (240 mm)
Mombasa   wettest month: May (240 mm)
Machakos  wettest month: Nov (182 mm)


## Task 5: Dry Months per County

**Approach:** The comparison `rain < 50` creates a True/False mask, and `sum(axis=1)` counts the True values in each row.

In [6]:
dry_mask = rain < 50
dry_counts = dry_mask.sum(axis=1)

for county, count in zip(counties, dry_counts):
    print(f"{county:<9} dry months (below 50 mm): {count}")

Nairobi   dry months (below 50 mm): 4
Nakuru    dry months (below 50 mm): 3
Kisumu    dry months (below 50 mm): 0
Mombasa   dry months (below 50 mm): 2
Machakos  dry months (below 50 mm): 6


## Task 6: Planting Window (Best Two Consecutive Months)

**Approach:** Adding the array to a shifted copy of itself (`rain[:, :-1] + rain[:, 1:]`) gives the combined rainfall of every pair of neighbouring months. `argmax(axis=1)` finds the best pair, and position `p` means the months at positions `p` and `p + 1`.

In [7]:
pair_totals = rain[:, :-1] + rain[:, 1:]
best_pos = pair_totals.argmax(axis=1)
best_mm = pair_totals.max(axis=1)
planting_windows = [f"{months[p]}-{months[p + 1]}" for p in best_pos]

for county, window, mm in zip(counties, planting_windows, best_mm):
    print(f"{county:<9} planting window: {window} ({mm} mm combined)")

Nairobi   planting window: Apr-May (395 mm combined)
Nakuru    planting window: Apr-May (265 mm combined)
Kisumu    planting window: Apr-May (420 mm combined)
Mombasa   planting window: Apr-May (390 mm combined)
Machakos  planting window: Nov-Dec (278 mm combined)


## Task 7: Write rainfall_summary.csv

**Approach:** Build one dictionary per county, write them with `csv.DictWriter` inside a `with` block, then read the file back with `csv.DictReader` to confirm it is readable.

In [8]:
import csv

summary_rows = []
for county, total, wet, dry, window in zip(
        counties, annual_totals, wettest_months, dry_counts, planting_windows):
    summary_rows.append({
        "county": county,
        "annual_total_mm": int(total),
        "wettest_month": wet,
        "dry_months": int(dry),
        "planting_window": window,
    })

fieldnames = ["county", "annual_total_mm", "wettest_month", "dry_months", "planting_window"]
with open("rainfall_summary.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(summary_rows)

with open("rainfall_summary.csv", newline="") as f:
    for row in csv.DictReader(f):
        print(row)

{'county': 'Nairobi', 'annual_total_mm': '1045', 'wettest_month': 'Apr', 'dry_months': '4', 'planting_window': 'Apr-May'}
{'county': 'Nakuru', 'annual_total_mm': '916', 'wettest_month': 'Apr', 'dry_months': '3', 'planting_window': 'Apr-May'}
{'county': 'Kisumu', 'annual_total_mm': '1371', 'wettest_month': 'Apr', 'dry_months': '0', 'planting_window': 'Apr-May'}
{'county': 'Mombasa', 'annual_total_mm': '1075', 'wettest_month': 'May', 'dry_months': '2', 'planting_window': 'Apr-May'}
{'county': 'Machakos', 'annual_total_mm': '811', 'wettest_month': 'Nov', 'dry_months': '6', 'planting_window': 'Nov-Dec'}


## Task 8: Planting Advisory for the NGO

**Approach:** Each point below is backed by a number from Tasks 1 to 7, and the last sentence states the data-quality caveat.

## Advisory for the NGO

Send April to May planting advice to Nairobi, Nakuru, Kisumu and Mombasa, because that is the wettest two-month window in all four counties, with Kisumu highest at 420 mm combined. Advise Machakos separately to plant in November to December (278 mm combined), which beats its April to May total of 208 mm. Kisumu is the most reliable county, with 0 dry months and 1,371 mm for the year, while Machakos is the driest at 811 mm with 6 dry months, so it also needs drought advice. Mombasa and Nairobi each have dry spells (2 and 4 dry months), so farmers should plant at the start of their window rather than waiting. Please treat the Machakos figures with caution: 2 of its 12 readings (April and September) were invalid and were filled with the average of 68 mm, and only one year of data was used, so the Machakos advice should be confirmed with the station, although the November to December window would only lose its lead if the true April reading was above 138 mm.